# Clean Experiment: 6-PLD vs 4-PLD vs 3-PLD ASL-MRI DNN

**Purpose:** This notebook is the single authoritative experiment produced after
the forensic reproducibility audit. It trains, evaluates, and compares all three
PLD configurations from scratch using the corrected ASL simulation (λ bug fixed).

**Audit corrections applied:**
1. `paper_signal` in `src/simulation.py` no longer divides by λ twice.
2. All old results archived to `results/_archived_pre_audit/`.
3. All models retrained from scratch on corrected signals.
4. All evaluations use a single common test set.
5. Every metric is computed live from prediction arrays — nothing is hard-coded.


## 1. Experiment Configuration (Single Source of Truth)

In [1]:
import sys, os, copy, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import r2_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# ── Project root ──
project_root = Path.cwd().parent if not (Path.cwd() / "src").exists() else Path.cwd()
sys.path.insert(0, str(project_root / "src"))
from simulation import SimulationConfig, seed_everything, generate_dataset, reference_signal, noisy_signals

# ── Device ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ═══════════════════════════════════════════════════════════
# SINGLE SOURCE OF TRUTH — every parameter defined ONCE here
# ═══════════════════════════════════════════════════════════
EXPERIMENT_CONFIG = {
    # Random seeds
    "seed": 42,
    "train_seed": 52,      # SEED + 10
    "val_seed": 62,        # SEED + 20
    "test_seed": 999,

    # Dataset sizes
    "n_train": 4000,
    "n_val": 1000,
    "n_test": 10000,

    # Physics / simulation
    "cbf_range": [0.0, 100.0],
    "att_range_s": [0.5, 3.0],
    "tau_s": 1.8,
    "alpha": 0.85,
    "beta": 0.75,
    "lambda_blood": 0.9,
    "t1_tissue_s": 1.2,
    "t1_blood_s": 1.66,
    "scale": 100000.0,
    "snr": 10.0,

    # PLD configurations
    "plds_6": [1.525, 2.025, 2.525, 3.025, 3.525, 4.025],
    "plds_4_indices": [0, 1, 2, 3],
    "plds_3_indices": [0, 1, 3],

    # DNN architecture
    "n_hidden_layers": 9,  # input + 8 hidden + output = 10 layers total
    "cbf_width": 50,
    "att_width": 100,
    "activation": "ELU",

    # Training
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "batch_size": 512,
    "max_epochs": 200,
    "patience": 20,
    "loss": "L1Loss (MAE)",
    "grad_clip": 1.0,

    # SNR sweep
    "snr_sweep": list(range(10, 81, 5)),
}

cfg = SimulationConfig()
SEED = EXPERIMENT_CONFIG["seed"]

# Print reference signal to verify λ fix
ref_sig = reference_signal(cfg)
print(f"Reference signal (corrected, single λ): {ref_sig:.4f}")
print(f"Expected ratio vs old (334.2039): {ref_sig / 334.2039:.6f} ≈ 0.9")
print(f"Config loaded: {len(EXPERIMENT_CONFIG)} parameters defined.")


Device: cuda
Reference signal (corrected, single λ): 300.7835
Expected ratio vs old (334.2039): 0.900000 ≈ 0.9
Config loaded: 32 parameters defined.


## 2. Dataset Generation

In [2]:
seed_everything(SEED)
torch.manual_seed(SEED)

N_TRAIN = EXPERIMENT_CONFIG["n_train"]
N_VAL = EXPERIMENT_CONFIG["n_val"]
N_TEST = EXPERIMENT_CONFIG["n_test"]
SNR = EXPERIMENT_CONFIG["snr"]

# Generate datasets with independent seeds — NO overlap possible
train_full, Y_train, _ = generate_dataset(N_TRAIN, cfg, EXPERIMENT_CONFIG["train_seed"], snr=SNR)
val_full, Y_val, _ = generate_dataset(N_VAL, cfg, EXPERIMENT_CONFIG["val_seed"], snr=SNR)
test_full, Y_test, _ = generate_dataset(N_TEST, cfg, EXPERIMENT_CONFIG["test_seed"], snr=SNR)

# Verify no overlap
tr_set = set(map(tuple, Y_train))
va_set = set(map(tuple, Y_val))
te_set = set(map(tuple, Y_test))
assert len(tr_set & va_set) == 0, "LEAK: train/val overlap!"
assert len(tr_set & te_set) == 0, "LEAK: train/test overlap!"
assert len(va_set & te_set) == 0, "LEAK: val/test overlap!"

print(f"Train: {train_full.shape} targets: {Y_train.shape}")
print(f"Val:   {val_full.shape}  targets: {Y_val.shape}")
print(f"Test:  {test_full.shape} targets: {Y_test.shape}")
print(f"CBF range: [{Y_train[:,0].min():.4f}, {Y_train[:,0].max():.4f}]")
print(f"ATT range: [{Y_train[:,1].min():.4f}, {Y_train[:,1].max():.4f}]")
print("Data leakage check: PASSED (0 overlapping samples)")


Train: (4000, 6) targets: (4000, 2)
Val:   (1000, 6)  targets: (1000, 2)
Test:  (10000, 6) targets: (10000, 2)
CBF range: [0.0071, 99.8716]
ATT range: [0.5004, 2.9997]
Data leakage check: PASSED (0 overlapping samples)


## 3. DNN Architecture & Training Functions

In [3]:
class StandardizedNet(nn.Module):
    """Paper-matched MLP: input → 8 hidden ELU layers → 1 output.
    Width is 50 for CBF, 100 for ATT (Ishida et al. 2024, Table 1)."""
    def __init__(self, input_dim, width):
        super().__init__()
        layers = [nn.Linear(input_dim, width), nn.ELU()]
        for _ in range(8):
            layers += [nn.Linear(width, width), nn.ELU()]
        layers.append(nn.Linear(width, 1))
        self.backbone = nn.Sequential(*layers)
        for layer in self.backbone:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        return self.backbone(x)


def train_net(net, x_train, y_train, x_val, y_val, y_mean, y_std, name):
    """Train a StandardizedNet with early stopping. Returns training history DataFrame."""
    # Standardize targets using TRAINING statistics only
    yt = ((y_train - y_mean) / y_std).astype("float32")
    yv = ((y_val - y_mean) / y_std).astype("float32")

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_train), torch.from_numpy(yt[:, None])),
        batch_size=EXPERIMENT_CONFIG["batch_size"], shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_val), torch.from_numpy(yv[:, None])),
        batch_size=EXPERIMENT_CONFIG["batch_size"],
    )

    opt = torch.optim.Adam(net.parameters(), lr=EXPERIMENT_CONFIG["learning_rate"])
    loss_fn = nn.L1Loss()

    best_val = float("inf")
    best_state = copy.deepcopy(net.state_dict())
    stale = 0
    rows = []

    for epoch in range(1, EXPERIMENT_CONFIG["max_epochs"] + 1):
        # ── Training ──
        net.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = loss_fn(net(xb.to(device)), yb.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), EXPERIMENT_CONFIG["grad_clip"])
            opt.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # ── Validation ──
        net.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                val_loss += loss_fn(net(xb.to(device)), yb.to(device)).item()
        val_loss /= len(val_loader)

        rows.append({"epoch": epoch, "train_mae": train_loss, "validation_mae": val_loss})
        if epoch % 10 == 0 or epoch == 1:
            print(f"  {name} Epoch {epoch:03d}: Train={train_loss:.6f}  Val={val_loss:.6f}")

        # ── Early stopping ──
        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(net.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= EXPERIMENT_CONFIG["patience"]:
                print(f"  {name} early stopped at epoch {epoch} (best val={best_val:.6f})")
                break

    net.load_state_dict(best_state)
    return pd.DataFrame(rows)


def calc_metrics(y_true, y_pred):
    """Calculate all evaluation metrics from arrays. Nothing hard-coded."""
    e = y_pred - y_true
    return {
        "MAE": float(np.mean(np.abs(e))),
        "RMSE": float(np.sqrt(np.mean(e**2))),
        "R2": float(r2_score(y_true, y_pred)),
        "Pearson": float(np.corrcoef(y_true, y_pred)[0, 1]),
        "Bias": float(np.mean(e)),
        "Error_SD": float(np.std(e, ddof=1)),
    }


def predict(cbf_net, att_net, x_test, cbf_mean, cbf_std, att_mean, att_std):
    """Generate predictions and clamp to physical range."""
    cbf_net.eval()
    att_net.eval()
    with torch.no_grad():
        xt = torch.from_numpy(x_test).to(device)
        cbf_pred = cbf_net(xt).cpu().squeeze(1).numpy() * cbf_std + cbf_mean
        att_pred = att_net(xt).cpu().squeeze(1).numpy() * att_std + att_mean
    cbf_pred = np.clip(cbf_pred, *EXPERIMENT_CONFIG["cbf_range"])
    att_pred = np.clip(att_pred, *EXPERIMENT_CONFIG["att_range_s"])
    return np.column_stack((cbf_pred, att_pred))

print("Architecture & training functions defined.")
print(f"StandardizedNet: input → 8 hidden ELU layers → 1 output")


Architecture & training functions defined.
StandardizedNet: input → 8 hidden ELU layers → 1 output


## 4. Train All Models

Each PLD configuration is trained using the SAME:
- underlying data (sliced from common 6-PLD signals)
- target normalization statistics (from training targets)
- architecture (StandardizedNet with ELU, 50w CBF / 100w ATT)
- optimizer, learning rate, batch size, patience, loss

The ONLY difference is the number of input features (PLDs).


In [4]:
# ── Target normalization from TRAINING data only ──
CBF_mean = float(Y_train[:, 0].mean())
CBF_std = float(Y_train[:, 0].std() + 1e-8)
ATT_mean = float(Y_train[:, 1].mean())
ATT_std = float(Y_train[:, 1].std() + 1e-8)
print(f"Target stats (train only): CBF μ={CBF_mean:.4f} σ={CBF_std:.4f}, ATT μ={ATT_mean:.4f} σ={ATT_std:.4f}")

PLD_CONFIGS = {
    "6_pld": {"indices": list(range(6)), "input_dim": 6, "plds": EXPERIMENT_CONFIG["plds_6"]},
    "4_pld": {"indices": EXPERIMENT_CONFIG["plds_4_indices"], "input_dim": 4,
              "plds": [EXPERIMENT_CONFIG["plds_6"][i] for i in EXPERIMENT_CONFIG["plds_4_indices"]]},
    "3_pld": {"indices": EXPERIMENT_CONFIG["plds_3_indices"], "input_dim": 3,
              "plds": [EXPERIMENT_CONFIG["plds_6"][i] for i in EXPERIMENT_CONFIG["plds_3_indices"]]},
}

all_results = {}  # {name: {cbf_net, att_net, predictions, metrics, histories, norm_stats}}

for name, pcfg in PLD_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"  EXPERIMENT: {name.upper()}  (PLDs = {pcfg['plds']})")
    print(f"{'='*60}")

    idx = pcfg["indices"]
    dim = pcfg["input_dim"]

    # ── Slice inputs from common arrays ──
    X_tr = train_full[:, idx].copy()
    X_va = val_full[:, idx].copy()
    X_te = test_full[:, idx].copy()

    # ── Input normalization from TRAINING data only ──
    X_mean = X_tr.mean(0, keepdims=True).astype("float32")
    X_std = X_tr.std(0, keepdims=True).astype("float32") + 1e-8
    X_tr_n = ((X_tr - X_mean) / X_std).astype("float32")
    X_va_n = ((X_va - X_mean) / X_std).astype("float32")
    X_te_n = ((X_te - X_mean) / X_std).astype("float32")

    # ── Train CBF model ──
    print(f"\n  Training {name} CBF (width={EXPERIMENT_CONFIG['cbf_width']})...")
    cbf_net = StandardizedNet(dim, EXPERIMENT_CONFIG["cbf_width"]).to(device)
    cbf_hist = train_net(cbf_net, X_tr_n, Y_train[:, 0], X_va_n, Y_val[:, 0], CBF_mean, CBF_std, f"{name}-CBF")

    # ── Train ATT model ──
    print(f"\n  Training {name} ATT (width={EXPERIMENT_CONFIG['att_width']})...")
    att_net = StandardizedNet(dim, EXPERIMENT_CONFIG["att_width"]).to(device)
    att_hist = train_net(att_net, X_tr_n, Y_train[:, 1], X_va_n, Y_val[:, 1], ATT_mean, ATT_std, f"{name}-ATT")

    # ── Predict on common test set ──
    preds = predict(cbf_net, att_net, X_te_n, CBF_mean, CBF_std, ATT_mean, ATT_std)

    # ── Calculate metrics ──
    metrics = []
    for j, param in enumerate(["CBF", "ATT"]):
        row = {"Model": name, "Parameter": param}
        row.update(calc_metrics(Y_test[:, j], preds[:, j]))
        metrics.append(row)

    # ── Save everything ──
    out_dir = project_root / "results" / name
    out_dir.mkdir(parents=True, exist_ok=True)

    torch.save(cbf_net.state_dict(), out_dir / "cbf_model.pt")
    torch.save(att_net.state_dict(), out_dir / "att_model.pt")
    cbf_hist.to_csv(out_dir / "cbf_training_history.csv", index=False)
    att_hist.to_csv(out_dir / "att_training_history.csv", index=False)
    np.savez(out_dir / "test_predictions.npz", y_true=Y_test, y_pred=preds)
    np.savez(out_dir / "normalization.npz", X_mean=X_mean, X_std=X_std,
             CBF_mean=CBF_mean, CBF_std=CBF_std, ATT_mean=ATT_mean, ATT_std=ATT_std)
    pd.DataFrame(metrics).to_csv(out_dir / "metrics.csv", index=False)

    # ── Save provenance ──
    provenance = {
        "experiment": name,
        "plds": pcfg["plds"],
        "pld_indices": pcfg["indices"],
        "input_dim": dim,
        "cbf_width": EXPERIMENT_CONFIG["cbf_width"],
        "att_width": EXPERIMENT_CONFIG["att_width"],
        "n_train": N_TRAIN, "n_val": N_VAL, "n_test": N_TEST,
        "train_seed": EXPERIMENT_CONFIG["train_seed"],
        "val_seed": EXPERIMENT_CONFIG["val_seed"],
        "test_seed": EXPERIMENT_CONFIG["test_seed"],
        "snr": SNR,
        "cbf_range": EXPERIMENT_CONFIG["cbf_range"],
        "att_range_s": EXPERIMENT_CONFIG["att_range_s"],
        "optimizer": EXPERIMENT_CONFIG["optimizer"],
        "learning_rate": EXPERIMENT_CONFIG["learning_rate"],
        "batch_size": EXPERIMENT_CONFIG["batch_size"],
        "max_epochs": EXPERIMENT_CONFIG["max_epochs"],
        "patience": EXPERIMENT_CONFIG["patience"],
        "loss": EXPERIMENT_CONFIG["loss"],
        "lambda_fix_applied": True,
        "cbf_best_epoch": int(cbf_hist.loc[cbf_hist["validation_mae"].idxmin(), "epoch"]),
        "att_best_epoch": int(att_hist.loc[att_hist["validation_mae"].idxmin(), "epoch"]),
    }
    with open(out_dir / "config.json", "w") as f:
        json.dump(provenance, f, indent=2)

    all_results[name] = {
        "cbf_net": cbf_net, "att_net": att_net,
        "predictions": preds, "metrics": metrics,
        "cbf_history": cbf_hist, "att_history": att_hist,
        "X_mean": X_mean, "X_std": X_std,
    }

    print(f"  Saved to: {out_dir}")

print("\n" + "="*60)
print("  ALL TRAINING COMPLETE")
print("="*60)


Target stats (train only): CBF μ=50.2351 σ=28.7727, ATT μ=1.7564 σ=0.7241

  EXPERIMENT: 6_PLD  (PLDs = [1.525, 2.025, 2.525, 3.025, 3.525, 4.025])

  Training 6_pld CBF (width=50)...


  6_pld-CBF Epoch 001: Train=0.745627  Val=0.371717


  6_pld-CBF Epoch 010: Train=0.135963  Val=0.135435


  6_pld-CBF Epoch 020: Train=0.129128  Val=0.148224


  6_pld-CBF Epoch 030: Train=0.130551  Val=0.151369


...

  3_pld-ATT Epoch 070: Train=0.353923  Val=0.389264


  3_pld-ATT Epoch 080: Train=0.351004  Val=0.402797


  3_pld-ATT Epoch 090: Train=0.347173  Val=0.382943


  3_pld-ATT Epoch 100: Train=0.347368  Val=0.378414
  3_pld-ATT early stopped at epoch 101 (best val=0.366763)


  Saved to: D:\fiinal year project\dnn_pld_reduction\results\3_pld

  ALL TRAINING COMPLETE


## 5. Comparison Table

In [5]:
# Build comparison from LIVE calculated metrics (not loaded from files)
all_metrics = []
for name in PLD_CONFIGS:
    all_metrics.extend(all_results[name]["metrics"])

comparison_df = pd.DataFrame(all_metrics)
comparison_df.to_csv(project_root / "results" / "comparison_metrics.csv", index=False)

display(Markdown("### Final Comparison (Corrected Simulation, SNR=10)"))
display(comparison_df.round(5))


### Final Comparison (Corrected Simulation, SNR=10)

,Model,Parameter,MAE,RMSE,R2,Pearson,Bias,Error_SD
0,6_pld,CBF,3.62619,4.87181,0.97153,0.98592,-0.51404,4.84485
1,6_pld,ATT,0.25596,0.38957,0.70652,0.84592,0.06128,0.38474
2,4_pld,CBF,3.54338,4.82568,0.97207,0.98608,0.23200,4.82034
3,4_pld,ATT,0.25071,0.38289,0.71650,0.85013,0.05399,0.37909
4,3_pld,CBF,4.07556,5.62202,0.96209,0.98089,-0.20311,5.61863
5,3_pld,ATT,0.26227,0.39913,0.69194,0.83755,0.05080,0.39591


## 6. Independent Metric Verification

In [6]:
# Reload predictions from disk and recalculate metrics independently
print("Verifying saved metrics against independent recalculation from disk...")
all_ok = True
for name in PLD_CONFIGS:
    npz = np.load(project_root / "results" / name / "test_predictions.npz")
    csv = pd.read_csv(project_root / "results" / name / "metrics.csv")
    y_true_disk = npz["y_true"]
    y_pred_disk = npz["y_pred"]

    for j, param in enumerate(["CBF", "ATT"]):
        recalc = calc_metrics(y_true_disk[:, j], y_pred_disk[:, j])
        reported = csv[csv.Parameter == param].iloc[0].to_dict()
        for met in ["MAE", "RMSE", "R2", "Pearson", "Bias"]:
            diff = abs(recalc[met] - reported[met])
            status = "OK" if diff < 1e-6 else "MISMATCH"
            if status == "MISMATCH":
                all_ok = False
            print(f"  {name} {param} {met}: reported={reported[met]:.6f} recalc={recalc[met]:.6f} diff={diff:.2e} [{status}]")

if all_ok:
    print("\n✓ ALL METRICS VERIFIED — saved values match independent recalculation.")
else:
    print("\n✗ VERIFICATION FAILED — see MISMATCH entries above.")


Verifying saved metrics against independent recalculation from disk...
  6_pld CBF MAE: reported=3.626190 recalc=3.626190 diff=0.00e+00 [OK]
  6_pld CBF RMSE: reported=4.871808 recalc=4.871808 diff=0.00e+00 [OK]
  6_pld CBF R2: reported=0.971532 recalc=0.971532 diff=1.11e-16 [OK]
  6_pld CBF Pearson: reported=0.985918 recalc=0.985918 diff=0.00e+00 [OK]
  6_pld CBF Bias: reported=-0.514045 recalc=-0.514045 diff=0.00e+00 [OK]
  6_pld ATT MAE: reported=0.255960 recalc=0.255960 diff=0.00e+00 [OK]
  6_pld ATT RMSE: reported=0.389570 recalc=0.389570 diff=0.00e+00 [OK]
  6_pld ATT R2: reported=0.706525 recalc=0.706525 diff=0.00e+00 [OK]
  6_pld ATT Pearson: reported=0.845923 recalc=0.845923 diff=0.00e+00 [OK]
  6_pld ATT Bias: reported=0.061282 recalc=0.061282 diff=1.39e-17 [OK]
  4_pld CBF MAE: reported=3.543383 recalc=3.543383 diff=0.00e+00 [OK]
  4_pld CBF RMSE: reported=4.825679 recalc=4.825679 diff=0.00e+00 [OK]
  4_pld CBF R2: reported=0.972069 recalc=0.972069 diff=0.00e+00 [OK]
  4_pld

## 7. Checkpoint Re-inference Verification

In [7]:
# Reload checkpoints from disk, re-run inference, compare against saved predictions
print("Verifying checkpoint → prediction chain...")
for name, pcfg in PLD_CONFIGS.items():
    out_dir = project_root / "results" / name
    norm = np.load(out_dir / "normalization.npz")

    # Reload models from disk
    cbf_check = StandardizedNet(pcfg["input_dim"], EXPERIMENT_CONFIG["cbf_width"]).to(device)
    cbf_check.load_state_dict(torch.load(out_dir / "cbf_model.pt", map_location=device, weights_only=True))

    att_check = StandardizedNet(pcfg["input_dim"], EXPERIMENT_CONFIG["att_width"]).to(device)
    att_check.load_state_dict(torch.load(out_dir / "att_model.pt", map_location=device, weights_only=True))

    # Re-normalize test inputs from scratch
    X_te = test_full[:, pcfg["indices"]]
    X_te_n = ((X_te - norm["X_mean"]) / norm["X_std"]).astype("float32")

    # Re-predict
    preds_check = predict(cbf_check, att_check, X_te_n,
                          float(norm["CBF_mean"]), float(norm["CBF_std"]),
                          float(norm["ATT_mean"]), float(norm["ATT_std"]))

    saved_preds = np.load(out_dir / "test_predictions.npz")["y_pred"]
    max_diff = np.max(np.abs(preds_check - saved_preds))
    status = "OK" if max_diff < 1e-4 else "MISMATCH"
    print(f"  {name}: checkpoint re-inference max diff = {max_diff:.2e} [{status}]")

print("\n✓ Checkpoint verification complete.")


Verifying checkpoint → prediction chain...


  6_pld: checkpoint re-inference max diff = 0.00e+00 [OK]


  4_pld: checkpoint re-inference max diff = 0.00e+00 [OK]


  3_pld: checkpoint re-inference max diff = 0.00e+00 [OK]

✓ Checkpoint verification complete.


## 8. Training Curves

In [8]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, name in enumerate(PLD_CONFIGS):
    for row, (param, hist_key) in enumerate([("CBF", "cbf_history"), ("ATT", "att_history")]):
        ax = axes[row, col]
        h = all_results[name][hist_key]
        ax.plot(h["epoch"], h["train_mae"], label="Train MAE")
        ax.plot(h["epoch"], h["validation_mae"], label="Val MAE")
        best_ep = h.loc[h["validation_mae"].idxmin(), "epoch"]
        ax.axvline(best_ep, color="red", linestyle="--", alpha=0.5, label=f"Best epoch={int(best_ep)}")
        ax.set_title(f"{name} {param}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("MAE")
        ax.legend(fontsize=7)

fig.tight_layout()
fig.savefig(project_root / "results" / "training_curves.png", dpi=150)
plt.show()
print("Saved: results/training_curves.png")


Saved: results/training_curves.png


C:\Users\ANU THOMSON\AppData\Local\Temp\ipykernel_25208\978715473.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Prediction Scatter Plots

In [9]:
for j, param in enumerate(["CBF", "ATT"]):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for col, name in enumerate(PLD_CONFIGS):
        ax = axes[col]
        preds = all_results[name]["predictions"]
        ax.scatter(Y_test[:, j], preds[:, j], s=3, alpha=0.2)
        lo, hi = Y_test[:, j].min(), Y_test[:, j].max()
        ax.plot([lo, hi], [lo, hi], "k--", linewidth=1)
        m = all_results[name]["metrics"][j]
        ax.set_title(f"{name} {param}\nMAE={m['MAE']:.3f} R²={m['R2']:.4f}")
        ax.set_xlabel(f"True {param}")
        ax.set_ylabel(f"Predicted {param}")
    fig.tight_layout()
    fig.savefig(project_root / "results" / f"scatter_{param.lower()}.png", dpi=150)
    plt.show()
print("Saved: results/scatter_cbf.png, results/scatter_att.png")


C:\Users\ANU THOMSON\AppData\Local\Temp\ipykernel_25208\3191410229.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: results/scatter_cbf.png, results/scatter_att.png


## 10. SNR Robustness Sweep (10 to 80)

In [10]:
SNR_LEVELS = EXPERIMENT_CONFIG["snr_sweep"]
snr_rows = []

for snr_val in SNR_LEVELS:
    # Generate noisy test signals at this SNR using SAME underlying CBF/ATT
    snr_arr = np.full(len(Y_test), float(snr_val), dtype=np.float32)
    noisy_full = noisy_signals(
        Y_test[:, 0], Y_test[:, 1], snr_arr, cfg,
        rng=np.random.default_rng(10000 + snr_val),
    )

    for name, pcfg in PLD_CONFIGS.items():
        X_snr = noisy_full[:, pcfg["indices"]]
        r = all_results[name]
        X_snr_n = ((X_snr - r["X_mean"]) / r["X_std"]).astype("float32")
        preds = predict(r["cbf_net"], r["att_net"], X_snr_n,
                        CBF_mean, CBF_std, ATT_mean, ATT_std)

        for j, param in enumerate(["CBF", "ATT"]):
            m = calc_metrics(Y_test[:, j], preds[:, j])
            snr_rows.append({"SNR": snr_val, "Model": name, "Parameter": param, **m})

snr_df = pd.DataFrame(snr_rows)
snr_df.to_csv(project_root / "results" / "snr_sweep_metrics.csv", index=False)
print(f"SNR sweep complete: {len(snr_df)} rows saved.")

# Display pivoted tables
for param in ["CBF", "ATT"]:
    display(Markdown(f"### {param} MAE vs SNR"))
    display(snr_df[snr_df.Parameter == param].pivot(index="SNR", columns="Model", values="MAE").round(5))


SNR sweep complete: 90 rows saved.


### CBF MAE vs SNR

Model,3_pld,4_pld,6_pld
SNR,,,
10,3.76216,3.37313,3.39691
15,3.08131,2.73296,2.87830
20,2.83678,2.46489,2.65582
25,2.70204,2.32114,2.53622
30,2.63524,2.24635,2.47696
35,2.58834,2.19129,2.45489
40,2.55338,2.13299,2.39542
45,2.52491,2.12115,2.38315
50,2.51567,2.10014,2.37252


### ATT MAE vs SNR

Model,3_pld,4_pld,6_pld
SNR,,,
10,0.26118,0.25151,0.26148
15,0.22348,0.21622,0.22154
20,0.20475,0.20053,0.20085
25,0.19645,0.19354,0.19015
30,0.19001,0.18641,0.18232
35,0.18680,0.18328,0.17704
40,0.18277,0.18140,0.17332
45,0.18232,0.18029,0.17253
50,0.17973,0.17848,0.16861


## 11. SNR Sweep Plots

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for j, (param, ax) in enumerate(zip(["CBF", "ATT"], axes)):
    sub = snr_df[snr_df.Parameter == param]
    for name in PLD_CONFIGS:
        m = sub[sub.Model == name]
        ax.plot(m["SNR"], m["MAE"], "o-", label=name, markersize=4)
    ax.set_xlabel("SNR")
    ax.set_ylabel("MAE")
    ax.set_title(f"{param} MAE vs SNR")
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(project_root / "results" / "snr_sweep_plot.png", dpi=150)
plt.show()
print("Saved: results/snr_sweep_plot.png")


Saved: results/snr_sweep_plot.png


C:\Users\ANU THOMSON\AppData\Local\Temp\ipykernel_25208\3828082810.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Relative Change vs 6-PLD Baseline

In [12]:
relative_rows = []
for param in ["CBF", "ATT"]:
    base = comparison_df[(comparison_df.Model == "6_pld") & (comparison_df.Parameter == param)].iloc[0]
    for name in ["4_pld", "3_pld"]:
        reduced = comparison_df[(comparison_df.Model == name) & (comparison_df.Parameter == param)].iloc[0]
        relative_rows.append({
            "Model": name, "Parameter": param,
            "MAE_change_%": (reduced.MAE - base.MAE) / base.MAE * 100,
            "RMSE_change_%": (reduced.RMSE - base.RMSE) / base.RMSE * 100,
            "R2_change_%": (reduced.R2 - base.R2) / max(abs(base.R2), 1e-8) * 100,
        })

rel_df = pd.DataFrame(relative_rows)
rel_df.to_csv(project_root / "results" / "relative_change.csv", index=False)
display(Markdown("### Relative Change vs 6-PLD Baseline"))
display(rel_df.round(3))


### Relative Change vs 6-PLD Baseline

,Model,Parameter,MAE_change_%,RMSE_change_%,R2_change_%
0,4_pld,CBF,-2.284,-0.947,0.055
1,3_pld,CBF,12.392,15.399,-0.972
2,4_pld,ATT,-2.050,-1.714,1.412
3,3_pld,ATT,2.466,2.455,-2.065


## 13. Experiment Summary

In [13]:
print("="*60)
print("  EXPERIMENT SUMMARY")
print("="*60)
print(f"Simulation: Buxton single-compartment (λ bug FIXED)")
print(f"Reference signal: {reference_signal(cfg):.4f}")
print(f"Train: {N_TRAIN} | Val: {N_VAL} | Test: {N_TEST} | SNR: {SNR}")
print(f"Architecture: StandardizedNet (ELU, 9 hidden layers)")
print(f"  CBF: width={EXPERIMENT_CONFIG['cbf_width']}, ATT: width={EXPERIMENT_CONFIG['att_width']}")
print(f"Optimizer: Adam lr={EXPERIMENT_CONFIG['learning_rate']}")
print(f"Loss: L1 (MAE) | Batch: {EXPERIMENT_CONFIG['batch_size']} | Patience: {EXPERIMENT_CONFIG['patience']}")
print()

for name in PLD_CONFIGS:
    m = all_results[name]["metrics"]
    cbf_m = [x for x in m if x["Parameter"] == "CBF"][0]
    att_m = [x for x in m if x["Parameter"] == "ATT"][0]
    print(f"{name.upper()}:")
    print(f"  CBF: MAE={cbf_m['MAE']:.4f}  RMSE={cbf_m['RMSE']:.4f}  R²={cbf_m['R2']:.4f}")
    print(f"  ATT: MAE={att_m['MAE']:.4f}  RMSE={att_m['RMSE']:.4f}  R²={att_m['R2']:.4f}")

print()
print("All metrics verified against independent recalculation: ✓")
print("All checkpoints verified via re-inference: ✓")
print("Zero data leakage confirmed: ✓")
print("Common test set used for all models: ✓")


  EXPERIMENT SUMMARY
Simulation: Buxton single-compartment (λ bug FIXED)
Reference signal: 300.7835
Train: 4000 | Val: 1000 | Test: 10000 | SNR: 10.0
Architecture: StandardizedNet (ELU, 9 hidden layers)
  CBF: width=50, ATT: width=100
Optimizer: Adam lr=0.001
Loss: L1 (MAE) | Batch: 512 | Patience: 20

6_PLD:
  CBF: MAE=3.6262  RMSE=4.8718  R²=0.9715
  ATT: MAE=0.2560  RMSE=0.3896  R²=0.7065
4_PLD:
  CBF: MAE=3.5434  RMSE=4.8257  R²=0.9721
  ATT: MAE=0.2507  RMSE=0.3829  R²=0.7165
3_PLD:
  CBF: MAE=4.0756  RMSE=5.6220  R²=0.9621
  ATT: MAE=0.2623  RMSE=0.3991  R²=0.6919

All metrics verified against independent recalculation: ✓
All checkpoints verified via re-inference: ✓
Zero data leakage confirmed: ✓
Common test set used for all models: ✓
